In [16]:
import re
import math

PASSWORD_COMUNI = [
    "123456", "password", "123456789", "12345678", "12345",
    "1234567", "qwerty", "abc123", "football", "monkey",
    "letmein", "shadow", "master", "dragon", "111111",
    "baseball", "iloveyou", "trustno1", "sunshine", "princess",
    "welcome", "admin", "login", "solo", "password1"
]

def controlla_lunghezza(password):
    lunghezza = len(password)
    if lunghezza < 8:
        return 0, f"Troppo corta ({lunghezza} caratteri - minimo 8)"
    elif lunghezza < 12:
        return 1, f"Accettabile ({lunghezza} caratteri - meglio 12+)"
    elif lunghezza < 16:
        return 2, f"Buona ({lunghezza} caratteri)"
    else:
        return 3, f"Eccellente ({lunghezza} caratteri)"

def controlla_complessita(password):
    punteggio = 0
    feedback = []
    if re.search(r'[a-z]', password):
        punteggio += 1
    else:
        feedback.append("aggiungi lettere minuscole")
    if re.search(r'[A-Z]', password):
        punteggio += 1
    else:
        feedback.append("aggiungi lettere maiuscole")
    if re.search(r'[0-9]', password):
        punteggio += 1
    else:
        feedback.append("aggiungi numeri")
    if re.search(r'[!@#$%^&*()\[\],.{}|<>\-_]', password):
        punteggio += 1
    else:
        feedback.append("aggiungi simboli (!@#$...)")
    return punteggio, feedback

def controlla_pattern(password):
    penalita = 0
    feedback = []
    if re.search(r'(.)\1{2,}', password):
        penalita += 1
        feedback.append("evita caratteri ripetuti (aaa, 111...)")
    if re.search(r'(012|123|234|345|456|567|678|789|abc|bcd|cde|def)', password.lower()):
        penalita += 1
        feedback.append("evita sequenze ovvie (123, abc...)")
    if re.search(r'(19|20)\d{2}', password):
        penalita += 1
        feedback.append("evita anni (1990, 2024...)")
    return penalita, feedback

def controlla_password_comune(password):
    if password.lower() in PASSWORD_COMUNI:
        return True, "Password nella lista delle piu violate - cambiarla subito!"
    return False, ""

def calcola_entropia(password):
    pool = 0
    if re.search(r'[a-z]', password):
        pool += 26
    if re.search(r'[A-Z]', password):
        pool += 26
    if re.search(r'[0-9]', password):
        pool += 10
    if re.search(r'[!@#$%^&*()\[\],.{}|<>\-_]', password):
        pool += 32
    if pool == 0:
        return 0, "N/A"
    entropia = len(password) * math.log2(pool)
    if entropia < 28:
        livello = "Molto debole"
    elif entropia < 36:
        livello = "Debole"
    elif entropia < 60:
        livello = "Ragionevole"
    elif entropia < 128:
        livello = "Forte"
    else:
        livello = "Molto forte"
    return entropia, livello

def analizza_password(password):
    print("=" * 55)
    print(f"  ANALISI PASSWORD: {'*' * len(password)}")
    print("=" * 55)
    punteggio_totale = 0
    tutti_feedback = []

    e_comune, msg_comune = controlla_password_comune(password)
    if e_comune:
        print(f"\n  ATTENZIONE: {msg_comune}")
        print("=" * 55)
        return

    punti_lung, msg_lung = controlla_lunghezza(password)
    punteggio_totale += punti_lung
    print(f"\n  Lunghezza      : {msg_lung}")

    punti_comp, fb_comp = controlla_complessita(password)
    punteggio_totale += punti_comp
    tutti_feedback.extend(fb_comp)
    print(f"  Complessita    : {punti_comp}/4 categorie presenti")

    penalita, fb_pattern = controlla_pattern(password)
    punteggio_totale -= penalita
    tutti_feedback.extend(fb_pattern)
    if penalita > 0:
        print(f"  Pattern        : {penalita} pattern rischiosi trovati")
    else:
        print(f"  Pattern        : nessun pattern rischioso")

    entropia, livello_entropia = calcola_entropia(password)
    print(f"  Entropia       : {entropia:.1f} bit ({livello_entropia})")

    punteggio_totale = max(0, min(10, punteggio_totale * 10 // 7))
    if punteggio_totale <= 3:
        giudizio = "DEBOLE"
    elif punteggio_totale <= 6:
        giudizio = "MEDIA"
    elif punteggio_totale <= 8:
        giudizio = "FORTE"
    else:
        giudizio = "MOLTO FORTE"

    print(f"\n  Punteggio      : {punteggio_totale}/10 - {giudizio}")

    if tutti_feedback:
        print(f"\n  Consigli:")
        for consiglio in tutti_feedback:
            print(f"    -> {consiglio}")

    print("=" * 55)

# ── Test ──────────────────────────────────────────────
password_di_test = [
    "123456",
    "ciao",
    "password123",
    "Marco1990!",
    "X7$kpL!mQ2",
    "Tr0ub4dor3",
]

for pwd in password_di_test:
    analizza_password(pwd)
    print()


  ANALISI PASSWORD: ******

  ATTENZIONE: Password nella lista delle piu violate - cambiarla subito!

  ANALISI PASSWORD: ****

  Lunghezza      : Troppo corta (4 caratteri - minimo 8)
  Complessita    : 1/4 categorie presenti
  Pattern        : nessun pattern rischioso
  Entropia       : 18.8 bit (Molto debole)

  Punteggio      : 1/10 - DEBOLE

  Consigli:
    -> aggiungi lettere maiuscole
    -> aggiungi numeri
    -> aggiungi simboli (!@#$...)

  ANALISI PASSWORD: ***********

  Lunghezza      : Accettabile (11 caratteri - meglio 12+)
  Complessita    : 2/4 categorie presenti
  Pattern        : 1 pattern rischiosi trovati
  Entropia       : 56.9 bit (Ragionevole)

  Punteggio      : 2/10 - DEBOLE

  Consigli:
    -> aggiungi lettere maiuscole
    -> aggiungi simboli (!@#$...)
    -> evita sequenze ovvie (123, abc...)

  ANALISI PASSWORD: **********

  Lunghezza      : Accettabile (10 caratteri - meglio 12+)
  Complessita    : 4/4 categorie presenti
  Pattern        : 1 pattern risc